In [ ]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version="2025-03-01-preview",
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

## 1. DataBase MCP

In [ ]:
db_path = '../data/w3schools'
with open(f'{db_path}/metadata.txt', 'r') as f:
    metadata = f.read()

In [ ]:
client = MultiServerMCPClient({
    "sqlite": {
        "transport": "stdio",
        "command": "/anaconda/envs/azureml_py38/bin/npx",
        "args": ["-y", "mcp-server-sqlite-npx", db_path + '/w3schools.db'],
    }
})


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SQL_SYSTEM = """당신은 SQLite 데이터베이스를 조회하는 어시스턴트입니다.
오직 제공된 MCP 도구(테이블 목록, 스키마, 쿼리 실행 등)를 사용해 DB만 조회하고, DB 외 정보는 제시하지 마세요.
- SELECT 위주로 실행하고, 필요 시 LIMIT을 두세요.
- 쿼리문은 답변 앞에 출력하세요.
- 결과는 요약 후 간단히 설명하세요. 응답은 한국어로 하세요.

[DATA DICTIONARY]
{metadata}
""".format(metadata=metadata)

tools = await client.get_tools()

agent = create_agent(
    llm,
    tools,
    system_prompt=SQL_SYSTEM,
    checkpointer=InMemorySaver(),
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "전체 고객 수와 총 주문 수를 각각 알려줘."}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content)

## 2. Sequential Thinking MCP

Sequential Thinking MCP는 복잡한 문제를 **단계별 사고(thought)** 로 쪼개서 풀 수 있게 해서 계획 수립, 디버깅, 전략 수립 같은 한번에 처리하기 힘든 문제를 분해해 순차적으로 해결할 수 있게 합니다.


In [ ]:
# Sequential Thinking MCP 서버만 사용하는 클라이언트
# npx -y @modelcontextprotocol/server-sequential-thinking
client_seq = MultiServerMCPClient({
    "sequential-thinking": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-sequential-thinking"],
    }
})

In [ ]:
# Sequential Thinking 도구를 에이전트에 연결해 단계별 계획/분석 요청
SEQ_SYSTEM = """당신은 복잡한 과제를 단계별로 정리하는 어시스턴트입니다.
반드시 sequential_thinking 도구를 사용해 사고 단계(thought)를 순서대로 문제를 해결하세요.
복잡한 계획·분석·디버깅 요청이 오면 이 도구로 단계를 쪼갠 뒤, 한국어로 답하세요."""

tools = await client_seq.get_tools()
agent = create_agent(
    llm,
    tools,
    system_prompt=SEQ_SYSTEM,
    checkpointer=InMemorySaver(),
)

query = "MCP와 A2A 에이전트를 통해 일정 관리 에이전트를 만들기 위한 계획을 작성해줘"
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content)

In [ ]:
SYSTEM = """당신은 복잡한 과제를 단계별로 정리하는 어시스턴트입니다.
반드시 사고 단계(thought)를 순서대로 계획하고 문제를 해결하세요.
복잡한 계획·분석·디버깅 요청이 오면 이 도구로 단계를 쪼갠 뒤, 한국어로 답하세요."""

simple_agent = create_agent(
    llm,
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

result = simple_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content)

## 파일 입출력 mcp

In [ ]:
# npx -y @modelcontextprotocol/server-filesystem <디렉토리_경로>

import os

# 폴더가 없을 경우에 생성
os.makedirs("./mcp_data", exist_ok=True)

client_fs = MultiServerMCPClient({
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y", 
            "@modelcontextprotocol/server-filesystem", 
            os.path.abspath("./mcp_data") 
        ],
    }
})

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = """당신은 파일 입출력을 수행하는 어시스턴트입니다.
반드시 filesystem 도구를 사용해 파일 입출력을 수행하세요.
파일 입출력 요청이 오면 이 도구로 파일 입출력을 수행하세요.
파일명은 영어로 작성하세요.
중요: 저장 경로는 반드시 mcp_data 폴더 안이어야 합니다. 파일 경로는 항상 mcp_data/ 로 시작하세요. 예: mcp_data/greeting.txt"""

tools = await client_fs.get_tools()

agent = create_agent(
    llm,
    tools,
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "영어로 인사말을 간단하게 작성해서 txt 파일로 저장해줘"}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content) 

In [ ]:
tools

In [ ]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "폴더에 어떤 파일이 있는지 알려줘"}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content) 

In [ ]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "mcp_data 폴더 안에 있는 파일을 삭제해줘"}]},
    config={"configurable": {"thread_id": "user1"}},
)
print(result["messages"][-1].content) 

In [ ]:
# https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop
import os
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from langchain.tools import tool

DATA_DIR = "./mcp_data"
@tool
def delete_file(filename: str) -> str:
    """
    agent_data 폴더 내의 특정 파일을 삭제합니다.
    불필요해진 임시 파일이나 잘못된 데이터 파일을 제거할 때 사용합니다.
    - filename: 삭제할 파일 이름 (예: 'temp_data.txt')
    """
    try:
        name = filename.strip().replace("mcp_data/", "").replace("mcp_data\\\\", "").lstrip("/\\\\")
        file_path = os.path.normpath(os.path.join(DATA_DIR, name))
        data_dir_abs = os.path.realpath(os.path.abspath(DATA_DIR))
        if not os.path.realpath(file_path).startswith(data_dir_abs):
            return "오류: mcp_data 폴더 밖의 경로는 삭제할 수 없습니다."
        if not os.path.exists(file_path):
            return f"오류: '{filename}' 파일을 찾을 수 없어 삭제에 실패했습니다."
        if os.path.isdir(file_path):
            return "오류: 디렉터리는 삭제할 수 없습니다. 파일만 지정하세요."
        os.remove(file_path)
        return f"성공적으로 '{filename}' 파일을 삭제했습니다."
    except Exception as e:
        return f"파일 삭제 중 오류 발생: {str(e)}"
    
added_tools = tools + [delete_file]

SYSTEM = """당신은 파일 입출력을 수행하는 어시스턴트입니다.
반드시 filesystem 도구를 사용해 파일 입출력을 수행하세요.
파일 입출력 요청이 오면 이 도구로 파일 입출력을 수행하세요.
파일명은 영어로 작성하세요.
파일을 삭제하기 전에는 삭제 요청을 확인하세요. ["approve", "edit", "reject"].
중요: 저장 경로는 반드시 mcp_data 폴더 안이어야 합니다. 파일 경로는 항상 mcp_data/ 로 시작하세요. 예: mcp_data/greeting.txt"""


agent_safe = create_agent(
    llm,
    added_tools,
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
            }
        ),
    ],
)

result_safe = await agent_safe.ainvoke(
    {"messages": [{"role": "user", "content": "안녕하세요? 라는 간단한 내용이 들어간 text 파일을 하나 생성해"}]},
    mode = 'update',
    config={"configurable": {"thread_id": "user1"}},
)
print(result_safe["messages"][-1].content)

# 삭제/이동 요청 시 에이전트가 도구를 호출하기 직전에 멈추고, 사용자 승인/수정/거부 후 재개됨
result_safe = await agent_safe.ainvoke(
    {"messages": [{"role": "user", "content": "방금 그 파일을 삭제해줘"}]},
    mode = 'update',
    config={"configurable": {"thread_id": "user1"}},
)
print(result_safe["messages"][-1].content)

In [ ]:
from langgraph.types import Command

result_final = await agent_safe.ainvoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ),
    config={"configurable": {"thread_id": "user1"}},
)

print(result_final["messages"][-1].content)